In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

***1. U Net with Residual layers***

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score, jaccard_score
from sklearn.utils.class_weight import compute_class_weight

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configuration
IMG_HEIGHT = 256
IMG_WIDTH = 256
BATCH_SIZE = 8
EPOCHS = 100
PATIENCE = 15

# Paths to your data
TRAIN_IMAGE_DIR = '/kaggle/input/microplastic-s/training/training/Training_original'
TRAIN_MASK_DIR = '/kaggle/input/microplastic-s/training/training/Finalmasks'
VAL_IMAGE_DIR = '/kaggle/input/microplastic-s/val/val/Original'
VAL_MASK_DIR = '/kaggle/input/microplastic-s/val/val/val_100masks'
TEST_IMAGE_DIR = '/kaggle/input/microplastic-s/Testing/Testing/Original'
TEST_MASK_DIR = '/kaggle/input/microplastic-s/Testing/Testing/testing_100masks'

# 1. Enhanced Data Loading with Preprocessing
def preprocess_image(image):
    """Enhanced image preprocessing"""
    # Convert to uint8 for OpenCV operations
    image_uint8 = (image * 255).astype(np.uint8)
    
    # Contrast Limited Adaptive Histogram Equalization (CLAHE)
    if len(image_uint8.shape) == 3:
        lab = cv2.cvtColor(image_uint8, cv2.COLOR_RGB2LAB)
        lab_planes = list(cv2.split(lab))
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        lab_planes[0] = clahe.apply(lab_planes[0])
        lab = cv2.merge(lab_planes)
        image_uint8 = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    
    # Gaussian blur for noise reduction
    image_processed = cv2.GaussianBlur(image_uint8, (3, 3), 0)
    
    # Convert back to float32
    return image_processed.astype(np.float32) / 255.0

def enhance_microplastic_contrast(image, mask):
    """Enhance contrast specifically around microplastic regions"""
    if len(mask.shape) == 3:
        mask_binary = mask.squeeze() > 0.5
    else:
        mask_binary = mask > 0.5
    
    # Convert to uint8 for OpenCV
    image_uint8 = (image * 255).astype(np.uint8)
    
    # Apply sharpening to entire image
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(image_uint8, -1, kernel)
    
    # Convert back to float
    sharpened = sharpened.astype(np.float32) / 255.0
    image_float = image_uint8.astype(np.float32) / 255.0
    
    # Blend based on mask
    enhanced_image = np.where(mask_binary[..., np.newaxis], sharpened, image_float)
    
    return enhanced_image

def load_data_from_dirs(image_dir, mask_dir, preprocess=True, enhance_contrast=True):
    image_paths = sorted([os.path.join(image_dir, fname) for fname in os.listdir(image_dir) 
                         if fname.endswith(('.png', '.jpg', '.jpeg'))])
    mask_paths = sorted([os.path.join(mask_dir, fname) for fname in os.listdir(mask_dir) 
                        if fname.endswith(('.png', '.jpg', '.jpeg'))])
    
    images = []
    masks = []
    
    for img_path, mask_path in zip(image_paths, mask_paths):
        # Load image
        img = Image.open(img_path).convert('RGB')
        img = img.resize((IMG_WIDTH, IMG_HEIGHT))
        img_array = np.array(img, dtype=np.float32) / 255.0
        
        # Load mask - ensure proper binarization
        mask = Image.open(mask_path).convert('L')
        mask = mask.resize((IMG_WIDTH, IMG_HEIGHT))
        mask_array = np.array(mask, dtype=np.float32)
        mask_array = (mask_array > 0.5).astype(np.float32)  # Proper binarization
        mask_array = np.expand_dims(mask_array, axis=-1)
        
        # Apply preprocessing
        if preprocess:
            img_array = preprocess_image(img_array)
        
        # Enhance contrast around microplastics
        if enhance_contrast:
            img_array = enhance_microplastic_contrast(img_array, mask_array)
        
        images.append(img_array)
        masks.append(mask_array)
    
    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# Load data with preprocessing
print("Loading and preprocessing data...")
X_train, y_train = load_data_from_dirs(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR, preprocess=True, enhance_contrast=True)
X_val, y_val = load_data_from_dirs(VAL_IMAGE_DIR, VAL_MASK_DIR, preprocess=True, enhance_contrast=True)
X_test, y_test = load_data_from_dirs(TEST_IMAGE_DIR, TEST_MASK_DIR, preprocess=True, enhance_contrast=True)

print(f"Training data: {X_train.shape}, {y_train.shape}")
print(f"Positive pixels in training: {np.sum(y_train > 0) / y_train.size:.6f}")

# 2. Calculate class weights for imbalance
def calculate_class_weights(masks):
    masks_flat = masks.flatten()
    class_weights = compute_class_weight('balanced', classes=[0, 1], y=masks_flat)
    return {0: class_weights[0], 1: class_weights[1]}

class_weights = calculate_class_weights(y_train)
print(f"Class weights: {class_weights}")

# 3. Fixed U-Net Architecture with Proper Residual Connections
def residual_block(input_tensor, num_filters):
    """Simple residual block without downsampling"""
    # Main path
    x = layers.Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    x = layers.Conv2D(num_filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Shortcut path
    shortcut = input_tensor
    if input_tensor.shape[-1] != num_filters:
        shortcut = layers.Conv2D(num_filters, (1, 1), padding='same')(input_tensor)
        shortcut = layers.BatchNormalization()(shortcut)
    
    # Add shortcut to main path
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    return x

def conv_block(input_tensor, num_filters, use_residual=True):
    """Convolutional block with optional residual connection"""
    if use_residual:
        return residual_block(input_tensor, num_filters)
    else:
        x = layers.Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        x = layers.Conv2D(num_filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        return x

def encoder_block(input_tensor, num_filters, use_residual=True):
    """Encoder block with proper downsampling"""
    # Apply convolutional block
    x = conv_block(input_tensor, num_filters, use_residual=use_residual)
    
    # Downsample
    p = layers.MaxPooling2D((2, 2))(x)
    
    return x, p

def decoder_block(input_tensor, skip_features, num_filters, use_residual=True):
    """Decoder block with proper upsampling and dimension matching"""
    # Upsample to match skip connection dimensions
    x = layers.Conv2DTranspose(num_filters, (2, 2), strides=2, padding='same')(input_tensor)
    
    # Ensure skip connection has the same number of filters
    if skip_features.shape[-1] != num_filters:
        skip_features = layers.Conv2D(num_filters, (1, 1), padding='same')(skip_features)
    
    # Concatenate with skip connection
    x = layers.concatenate([x, skip_features])
    
    # Apply convolutional/residual block
    x = conv_block(x, num_filters, use_residual=use_residual)
    
    return x

def build_residual_unet(input_shape=(256, 256, 3), use_residual=True):
    """Build U-Net with residual connections - FIXED VERSION"""
    inputs = layers.Input(shape=input_shape)
    
    # Encoder
    s1, p1 = encoder_block(inputs, 64, use_residual=use_residual)      # 256×256 -> 128×128
    s2, p2 = encoder_block(p1, 128, use_residual=use_residual)         # 128×128 -> 64×64
    s3, p3 = encoder_block(p2, 256, use_residual=use_residual)         # 64×64 -> 32×32
    s4, p4 = encoder_block(p3, 512, use_residual=use_residual)         # 32×32 -> 16×16
    
    # Bridge
    b1 = conv_block(p4, 1024, use_residual=use_residual)               # 16×16
    
    # Decoder - dimensions will be automatically matched by Conv2DTranspose
    d1 = decoder_block(b1, s4, 512, use_residual=use_residual)         # 16×16 -> 32×32
    d2 = decoder_block(d1, s3, 256, use_residual=use_residual)         # 32×32 -> 64×64
    d3 = decoder_block(d2, s2, 128, use_residual=use_residual)         # 64×64 -> 128×128
    d4 = decoder_block(d3, s1, 64, use_residual=use_residual)          # 128×128 -> 256×256
    
    # Output
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(d4)
    
    model = models.Model(inputs=inputs, outputs=outputs, name='Residual_U-Net')
    return model

# 4. Modified loss function with class weighting
def weighted_bce_dice_loss(y_true, y_pred):
    # Calculate class weights for this batch
    y_true_flat = K.flatten(y_true)
    y_pred_flat = K.flatten(y_pred)
    
    # Binary cross entropy with class weighting
    bce = tf.keras.losses.binary_crossentropy(y_true_flat, y_pred_flat)
    
    # Apply class weights to BCE
    class_weight_0 = tf.constant(class_weights[0], dtype=tf.float32)
    class_weight_1 = tf.constant(class_weights[1], dtype=tf.float32)
    
    # Create weight tensor based on true labels
    weights = tf.where(tf.equal(y_true_flat, 1.0), class_weight_1, class_weight_0)
    weighted_bce = tf.reduce_mean(weights * bce)
    
    # Dice loss
    intersection = tf.reduce_sum(y_true_flat * y_pred_flat)
    union = tf.reduce_sum(y_true_flat) + tf.reduce_sum(y_pred_flat)
    dice_loss = 1.0 - (2.0 * intersection + 1.0) / (union + 1.0)
    
    return weighted_bce + dice_loss

def dice_coef(y_true, y_pred, smooth=1):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

# 5. Enhanced metrics with thresholding
def precision_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    # Threshold predictions
    y_pred_thresholded = tf.cast(y_pred > 0.5, tf.float32)
    
    true_positives = tf.reduce_sum(y_true * y_pred_thresholded)
    predicted_positives = tf.reduce_sum(y_pred_thresholded)
    
    precision = true_positives / (predicted_positives + tf.keras.backend.epsilon())
    return precision

def recall_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    # Threshold predictions
    y_pred_thresholded = tf.cast(y_pred > 0.5, tf.float32)
    
    true_positives = tf.reduce_sum(y_true * y_pred_thresholded)
    actual_positives = tf.reduce_sum(y_true)
    
    recall = true_positives / (actual_positives + tf.keras.backend.epsilon())
    return recall

def f1_metric(y_true, y_pred):
    prec = precision_metric(y_true, y_pred)
    rec = recall_metric(y_true, y_pred)
    return 2 * ((prec * rec) / (prec + rec + tf.keras.backend.epsilon()))

# 6. Build and compile model
print("Building Residual U-Net...")
model = build_residual_unet(use_residual=True)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=weighted_bce_dice_loss,
    metrics=['accuracy', dice_coef, precision_metric, recall_metric, f1_metric]
)

model.summary()

# 7. Simplified and Fixed Data Generator
class MicroplasticDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, y, batch_size=8, shuffle=True, augment=False):
        super().__init__()
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.indexes = np.arange(len(X))
        self.on_epoch_end()
    
    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))
    
    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        
        X_batch = self.X[indexes]
        y_batch = self.y[indexes]
        
        if self.augment:
            X_batch, y_batch = self.augment_batch(X_batch, y_batch)
        
        return X_batch, y_batch
    
    def augment_batch(self, X_batch, y_batch):
        augmented_X = []
        augmented_y = []
        
        for i in range(len(X_batch)):
            img = X_batch[i].copy()
            mask = y_batch[i].copy()
            
            # Random horizontal flip (50% chance)
            if np.random.random() > 0.5:
                img = np.fliplr(img)
                mask = np.fliplr(mask)
            
            # Random vertical flip (50% chance)
            if np.random.random() > 0.5:
                img = np.flipud(img)
                mask = np.flipud(mask)
            
            # Random rotation (50% chance)
            if np.random.random() > 0.5:
                k = np.random.randint(1, 4)  # 90, 180, or 270 degrees
                img = np.rot90(img, k)
                mask = np.rot90(mask, k)
            
            # Random brightness adjustment (50% chance)
            if np.random.random() > 0.5:
                factor = np.random.uniform(0.7, 1.3)
                img = np.clip(img * factor, 0, 1)
            
            # Random contrast adjustment (50% chance)
            if np.random.random() > 0.5:
                factor = np.random.uniform(0.7, 1.3)
                mean = np.mean(img, axis=(0, 1), keepdims=True)
                img = np.clip((img - mean) * factor + mean, 0, 1)
            
            # Random Gaussian noise (30% chance)
            if np.random.random() > 0.7:
                noise = np.random.normal(0, 0.02, img.shape)
                img = np.clip(img + noise, 0, 1)
            
            augmented_X.append(img)
            augmented_y.append(mask)
        
        return np.array(augmented_X), np.array(augmented_y)
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# 8. Training with fixed data generator
train_generator = MicroplasticDataGenerator(X_train, y_train, batch_size=BATCH_SIZE, 
                                          shuffle=True, augment=True)
val_generator = MicroplasticDataGenerator(X_val, y_val, batch_size=BATCH_SIZE, 
                                        shuffle=False, augment=False)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_dice_coef', 
        patience=PATIENCE, 
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_dice_coef', 
        factor=0.5, 
        patience=8, 
        min_lr=1e-7,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_residual_unet_model.h5', 
        monitor='val_dice_coef', 
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

print("Starting training...")
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks,
    verbose=1
)

# 9. Evaluation
print("Loading best model for evaluation...")
model = tf.keras.models.load_model('best_residual_unet_model.h5', custom_objects={
    'weighted_bce_dice_loss': weighted_bce_dice_loss,
    'dice_coef': dice_coef,
    'precision_metric': precision_metric,
    'recall_metric': recall_metric,
    'f1_metric': f1_metric
})

print("Validation Metrics:")
val_results = model.evaluate(X_val, y_val, verbose=0)
print(f"Loss: {val_results[0]:.4f}")
print(f"Accuracy: {val_results[1]:.4f}")
print(f"Dice Coefficient: {val_results[2]:.4f}")
print(f"Precision: {val_results[3]:.4f}")
print(f"Recall: {val_results[4]:.4f}")
print(f"F1 Score: {val_results[5]:.4f}")

print("\nTest Metrics:")
test_results = model.evaluate(X_test, y_test, verbose=0)
print(f"Loss: {test_results[0]:.4f}")
print(f"Accuracy: {test_results[1]:.4f}")
print(f"Dice Coefficient: {test_results[2]:.4f}")
print(f"Precision: {test_results[3]:.4f}")
print(f"Recall: {test_results[4]:.4f}")
print(f"F1 Score: {test_results[5]:.4f}")

# 10. Visualization functions
def visualize_results(images, masks, predictions, num_samples=5):
    fig, axes = plt.subplots(num_samples, 4, figsize=(20, 5*num_samples))
    
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        # Original image
        axes[i, 0].imshow(images[i])
        axes[i, 0].set_title('Original Image')
        axes[i, 0].axis('off')
        
        # Ground truth mask
        axes[i, 1].imshow(masks[i].squeeze(), cmap='gray')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        # Prediction
        axes[i, 2].imshow(predictions[i].squeeze(), cmap='gray')
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')
        
        # Overlay
        axes[i, 3].imshow(images[i])
        axes[i, 3].imshow(predictions[i].squeeze(), cmap='jet', alpha=0.5)
        axes[i, 3].set_title('Overlay')
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig('residual_unet_segmentation_results.png', dpi=300, bbox_inches='tight')
    plt.show()

# Generate predictions
print("Generating predictions...")
val_predictions = model.predict(X_val, verbose=0)
test_predictions = model.predict(X_test, verbose=0)

# Apply threshold
val_predictions_binary = (val_predictions > 0.5).astype(np.float32)
test_predictions_binary = (test_predictions > 0.5).astype(np.float32)

# Visualize results
print("Visualizing validation results...")
visualize_results(X_val, y_val, val_predictions_binary, num_samples=min(5, len(X_val)))

print("Visualizing test results...")
visualize_results(X_test, y_test, test_predictions_binary, num_samples=min(5, len(X_test)))

# 11. Plot training history
def plot_training_history(history):
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    metrics = ['loss', 'accuracy', 'dice_coef', 'precision_metric', 'recall_metric', 'f1_metric']
    titles = ['Loss', 'Accuracy', 'Dice Coefficient', 'Precision', 'Recall', 'F1 Score']
    
    for i, (metric, title) in enumerate(zip(metrics, titles)):
        row, col = i // 3, i % 3
        axes[row, col].plot(history.history[metric], label=f'Training {title}', linewidth=2)
        axes[row, col].plot(history.history[f'val_{metric}'], label=f'Validation {title}', linewidth=2)
        axes[row, col].set_title(f'{title} - Residual U-Net', fontsize=12, fontweight='bold')
        axes[row, col].set_xlabel('Epoch')
        axes[row, col].set_ylabel(title)
        axes[row, col].legend()
        axes[row, col].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('residual_unet_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history(history)

print("Residual U-Net with preprocessing completed successfully!")

***2. U net with Spectral and residual with attention***

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score, jaccard_score
from sklearn.utils.class_weight import compute_class_weight

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configuration
IMG_HEIGHT = 256
IMG_WIDTH = 256
BATCH_SIZE = 8
EPOCHS = 100
PATIENCE = 15

# Paths to your data
TRAIN_IMAGE_DIR = '/kaggle/input/microplastic-s/training/training/Training_original'
TRAIN_MASK_DIR = '/kaggle/input/microplastic-s/training/training/Finalmasks'
VAL_IMAGE_DIR = '/kaggle/input/microplastic-s/val/val/Original'
VAL_MASK_DIR = '/kaggle/input/microplastic-s/val/val/val_100masks'
TEST_IMAGE_DIR = '/kaggle/input/microplastic-s/Testing/Testing/Original'
TEST_MASK_DIR = '/kaggle/input/microplastic-s/Testing/Testing/testing_100masks'

# ============================================================================
# 1. COMPREHENSIVE PREPROCESSING FUNCTIONS (APPLIED BEFORE TRAINING)
# ============================================================================

def advanced_preprocess_image(image):
    """Enhanced preprocessing pipeline with multiple techniques"""
    if len(image.shape) == 3:
        image_uint8 = (image * 255).astype(np.uint8)
        
        # 1. CLAHE for contrast enhancement
        lab = cv2.cvtColor(image_uint8, cv2.COLOR_RGB2LAB)
        lab_planes = list(cv2.split(lab))
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        lab_planes[0] = clahe.apply(lab_planes[0])
        lab = cv2.merge(lab_planes)
        image_uint8 = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        
        # 2. Gaussian blur for noise reduction
        image_uint8 = cv2.GaussianBlur(image_uint8, (3, 3), 0)
        
        # 3. Median filtering for salt-and-pepper noise
        image_uint8 = cv2.medianBlur(image_uint8, 3)
        
        # Convert back to float
        image_processed = image_uint8.astype(np.float32) / 255.0
        
        # 4. Color normalization
        normalized = np.zeros_like(image_processed)
        for channel in range(3):
            channel_data = image_processed[:, :, channel]
            normalized[:, :, channel] = (channel_data - np.mean(channel_data)) / (np.std(channel_data) + 1e-8)
        
        # Adaptive normalization
        if np.std(normalized) > 0.1:
            normalized = (normalized - np.min(normalized)) / (np.max(normalized) - np.min(normalized))
        
        return np.clip(normalized, 0, 1)
    else:
        return image

def enhance_microplastic_contrast(image, mask):
    """Enhanced contrast specifically around microplastic regions"""
    if len(mask.shape) == 3:
        mask_binary = mask.squeeze() > 0.5
    else:
        mask_binary = mask > 0.5
    
    image_uint8 = (image * 255).astype(np.uint8)
    
    # Apply sharpening to entire image
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(image_uint8, -1, kernel)
    
    sharpened = sharpened.astype(np.float32) / 255.0
    image_float = image_uint8.astype(np.float32) / 255.0
    
    # Blend based on mask
    enhanced_image = np.where(mask_binary[..., np.newaxis], sharpened, image_float)
    
    return enhanced_image

# ============================================================================
# 2. SPECTRAL ATTENTION BLOCKS (FIXED)
# ============================================================================

class SpectralAttention(layers.Layer):
    """Spectral attention for frequency-aware feature processing"""
    def __init__(self, filters, reduction_ratio=16, **kwargs):
        super(SpectralAttention, self).__init__(**kwargs)
        self.filters = filters
        self.reduction_ratio = reduction_ratio
        
        self.global_avg_pool = layers.GlobalAveragePooling2D()
        self.global_max_pool = layers.GlobalMaxPooling2D()
        
    def build(self, input_shape):
        self.fc1 = layers.Dense(self.filters // self.reduction_ratio, activation='relu')
        self.fc2 = layers.Dense(self.filters, activation='sigmoid')
        super(SpectralAttention, self).build(input_shape)
        
    def call(self, inputs):
        # Channel attention via both avg and max pooling
        avg_pool = self.global_avg_pool(inputs)
        max_pool = self.global_max_pool(inputs)
        
        # Shared MLP
        avg_out = self.fc2(self.fc1(avg_pool))
        max_out = self.fc2(self.fc1(max_pool))
        
        # Combine attention maps
        channel_attention = tf.sigmoid(avg_out + max_out)
        channel_attention = tf.reshape(channel_attention, [-1, 1, 1, self.filters])
        
        return inputs * channel_attention
    
    def get_config(self):
        config = super(SpectralAttention, self).get_config()
        config.update({
            'filters': self.filters,
            'reduction_ratio': self.reduction_ratio
        })
        return config

# ============================================================================
# 3. ENHANCED RESIDUAL BLOCKS WITH SPECTRAL ATTENTION
# ============================================================================

def residual_block(input_tensor, num_filters):
    """Simple residual block without downsampling"""
    # Main path
    x = layers.Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    x = layers.Conv2D(num_filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Shortcut path
    shortcut = input_tensor
    if input_tensor.shape[-1] != num_filters:
        shortcut = layers.Conv2D(num_filters, (1, 1), padding='same')(input_tensor)
        shortcut = layers.BatchNormalization()(shortcut)
    
    # Add shortcut to main path
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    return x

def enhanced_residual_block(input_tensor, num_filters, use_spectral_attention=True):
    """Enhanced residual block with spectral attention"""
    # Main path
    x = layers.Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    x = layers.Conv2D(num_filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Apply spectral attention
    if use_spectral_attention:
        x = SpectralAttention(num_filters)(x)
    
    # Shortcut path
    shortcut = input_tensor
    if input_tensor.shape[-1] != num_filters:
        shortcut = layers.Conv2D(num_filters, (1, 1), padding='same')(input_tensor)
        shortcut = layers.BatchNormalization()(shortcut)
    
    # Add residual connection
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    return x

# ============================================================================
# 4. SIMPLIFIED U-NET ARCHITECTURE WITH NOVEL COMPONENTS
# ============================================================================

def build_enhanced_unet(input_shape=(256, 256, 3)):
    """Enhanced U-Net with spectral attention and residual connections"""
    inputs = layers.Input(shape=input_shape)
    
    # Encoder
    # Level 1
    x1 = layers.Conv2D(64, (3, 3), padding='same')(inputs)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Activation('relu')(x1)
    x1 = SpectralAttention(64)(x1)  # Spectral attention
    p1 = layers.MaxPooling2D((2, 2))(x1)
    
    # Level 2
    x2 = enhanced_residual_block(p1, 128, use_spectral_attention=True)
    p2 = layers.MaxPooling2D((2, 2))(x2)
    
    # Level 3
    x3 = enhanced_residual_block(p2, 256, use_spectral_attention=True)
    p3 = layers.MaxPooling2D((2, 2))(x3)
    
    # Level 4
    x4 = enhanced_residual_block(p3, 512, use_spectral_attention=True)
    p4 = layers.MaxPooling2D((2, 2))(x4)
    
    # Bridge
    bridge = enhanced_residual_block(p4, 1024, use_spectral_attention=True)
    
    # Decoder
    # Level 4 up
    u4 = layers.Conv2DTranspose(512, (2, 2), strides=2, padding='same')(bridge)
    u4 = layers.concatenate([u4, x4])
    u4 = enhanced_residual_block(u4, 512, use_spectral_attention=True)
    
    # Level 3 up
    u3 = layers.Conv2DTranspose(256, (2, 2), strides=2, padding='same')(u4)
    u3 = layers.concatenate([u3, x3])
    u3 = enhanced_residual_block(u3, 256, use_spectral_attention=True)
    
    # Level 2 up
    u2 = layers.Conv2DTranspose(128, (2, 2), strides=2, padding='same')(u3)
    u2 = layers.concatenate([u2, x2])
    u2 = enhanced_residual_block(u2, 128, use_spectral_attention=True)
    
    # Level 1 up
    u1 = layers.Conv2DTranspose(64, (2, 2), strides=2, padding='same')(u2)
    u1 = layers.concatenate([u1, x1])
    u1 = enhanced_residual_block(u1, 64, use_spectral_attention=True)
    
    # Output
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(u1)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='Enhanced_Spectral_UNet')
    return model

# ============================================================================
# 5. DATA LOADING WITH PREPROCESSING
# ============================================================================

def load_data_from_dirs(image_dir, mask_dir, preprocess=True, enhance_contrast=True):
    image_paths = sorted([os.path.join(image_dir, fname) for fname in os.listdir(image_dir) 
                         if fname.endswith(('.png', '.jpg', '.jpeg'))])
    mask_paths = sorted([os.path.join(mask_dir, fname) for fname in os.listdir(mask_dir) 
                        if fname.endswith(('.png', '.jpg', '.jpeg'))])
    
    images = []
    masks = []
    
    for img_path, mask_path in zip(image_paths, mask_paths):
        # Load image
        img = Image.open(img_path).convert('RGB')
        img = img.resize((IMG_WIDTH, IMG_HEIGHT))
        img_array = np.array(img, dtype=np.float32) / 255.0
        
        # Load mask - ensure proper binarization
        mask = Image.open(mask_path).convert('L')
        mask = mask.resize((IMG_WIDTH, IMG_HEIGHT))
        mask_array = np.array(mask, dtype=np.float32)
        mask_array = (mask_array > 0.5).astype(np.float32)
        mask_array = np.expand_dims(mask_array, axis=-1)
        
        # Apply preprocessing
        if preprocess:
            img_array = advanced_preprocess_image(img_array)
        
        # Enhance contrast around microplastics
        if enhance_contrast:
            img_array = enhance_microplastic_contrast(img_array, mask_array)
        
        images.append(img_array)
        masks.append(mask_array)
    
    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# Load data with preprocessing
print("Loading and preprocessing data...")
X_train, y_train = load_data_from_dirs(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR, preprocess=True, enhance_contrast=True)
X_val, y_val = load_data_from_dirs(VAL_IMAGE_DIR, VAL_MASK_DIR, preprocess=True, enhance_contrast=True)
X_test, y_test = load_data_from_dirs(TEST_IMAGE_DIR, TEST_MASK_DIR, preprocess=True, enhance_contrast=True)

print(f"Training data: {X_train.shape}, {y_train.shape}")
print(f"Positive pixels in training: {np.sum(y_train > 0) / y_train.size:.6f}")

# ============================================================================
# 6. CLASS WEIGHTS AND LOSS FUNCTIONS
# ============================================================================

def calculate_class_weights(masks):
    masks_flat = masks.flatten()
    class_weights = compute_class_weight('balanced', classes=[0, 1], y=masks_flat)
    return {0: class_weights[0], 1: class_weights[1]}

class_weights = calculate_class_weights(y_train)
print(f"Class weights: {class_weights}")

def weighted_bce_dice_loss(y_true, y_pred):
    # Calculate class weights for this batch
    y_true_flat = K.flatten(y_true)
    y_pred_flat = K.flatten(y_pred)
    
    # Binary cross entropy with class weighting
    bce = tf.keras.losses.binary_crossentropy(y_true_flat, y_pred_flat)
    
    # Apply class weights to BCE
    class_weight_0 = tf.constant(class_weights[0], dtype=tf.float32)
    class_weight_1 = tf.constant(class_weights[1], dtype=tf.float32)
    
    # Create weight tensor based on true labels
    weights = tf.where(tf.equal(y_true_flat, 1.0), class_weight_1, class_weight_0)
    weighted_bce = tf.reduce_mean(weights * bce)
    
    # Dice loss
    intersection = tf.reduce_sum(y_true_flat * y_pred_flat)
    union = tf.reduce_sum(y_true_flat) + tf.reduce_sum(y_pred_flat)
    dice_loss = 1.0 - (2.0 * intersection + 1.0) / (union + 1.0)
    
    return weighted_bce + dice_loss

def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

# Enhanced metrics with thresholding
def precision_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    # Threshold predictions
    y_pred_thresholded = tf.cast(y_pred > 0.5, tf.float32)
    
    true_positives = tf.reduce_sum(y_true * y_pred_thresholded)
    predicted_positives = tf.reduce_sum(y_pred_thresholded)
    
    precision = true_positives / (predicted_positives + tf.keras.backend.epsilon())
    return precision

def recall_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    # Threshold predictions
    y_pred_thresholded = tf.cast(y_pred > 0.5, tf.float32)
    
    true_positives = tf.reduce_sum(y_true * y_pred_thresholded)
    actual_positives = tf.reduce_sum(y_true)
    
    recall = true_positives / (actual_positives + tf.keras.backend.epsilon())
    return recall

def f1_metric(y_true, y_pred):
    prec = precision_metric(y_true, y_pred)
    rec = recall_metric(y_true, y_pred)
    return 2 * ((prec * rec) / (prec + rec + tf.keras.backend.epsilon()))

# ============================================================================
# 7. BUILD AND COMPILE THE ENHANCED MODEL
# ============================================================================

print("Building Enhanced Spectral U-Net...")
model = build_enhanced_unet()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=weighted_bce_dice_loss,
    metrics=['accuracy', dice_coef, precision_metric, recall_metric, f1_metric]
)

model.summary()

# ============================================================================
# 8. FIXED DATA GENERATOR
# ============================================================================

class MicroplasticDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, y, batch_size=8, shuffle=True, augment=False):
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.indexes = np.arange(len(X))
        self.on_epoch_end()
    
    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))
    
    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        
        X_batch = self.X[indexes]
        y_batch = self.y[indexes]
        
        if self.augment:
            X_batch, y_batch = self.augment_batch(X_batch, y_batch)
        
        return X_batch, y_batch
    
    def augment_batch(self, X_batch, y_batch):
        augmented_X = []
        augmented_y = []
        
        for i in range(len(X_batch)):
            img = X_batch[i].copy()
            mask = y_batch[i].copy()
            
            # Random horizontal flip (50% chance)
            if np.random.random() > 0.5:
                img = np.fliplr(img)
                mask = np.fliplr(mask)
            
            # Random vertical flip (50% chance)
            if np.random.random() > 0.5:
                img = np.flipud(img)
                mask = np.flipud(mask)
            
            # Random rotation (50% chance)
            if np.random.random() > 0.5:
                k = np.random.randint(1, 4)  # 90, 180, or 270 degrees
                img = np.rot90(img, k)
                mask = np.rot90(mask, k)
            
            # Random brightness adjustment (50% chance)
            if np.random.random() > 0.5:
                factor = np.random.uniform(0.7, 1.3)
                img = np.clip(img * factor, 0, 1)
            
            # Random contrast adjustment (50% chance)
            if np.random.random() > 0.5:
                factor = np.random.uniform(0.7, 1.3)
                mean = np.mean(img, axis=(0, 1), keepdims=True)
                img = np.clip((img - mean) * factor + mean, 0, 1)
            
            # Random Gaussian noise (30% chance)
            if np.random.random() > 0.7:
                noise = np.random.normal(0, 0.02, img.shape)
                img = np.clip(img + noise, 0, 1)
            
            augmented_X.append(img)
            augmented_y.append(mask)
        
        return np.array(augmented_X), np.array(augmented_y)
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# ============================================================================
# 9. TRAINING WITH EARLY STOPPING
# ============================================================================

train_generator = MicroplasticDataGenerator(X_train, y_train, batch_size=BATCH_SIZE, 
                                          shuffle=True, augment=True)
val_generator = MicroplasticDataGenerator(X_val, y_val, batch_size=BATCH_SIZE, 
                                        shuffle=False, augment=False)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_dice_coef', 
        patience=PATIENCE, 
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_dice_coef', 
        factor=0.5, 
        patience=8, 
        min_lr=1e-7,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_enhanced_unet_model.h5', 
        monitor='val_dice_coef', 
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

print("Starting training...")
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks,
    verbose=1
)

# ============================================================================
# 10. EVALUATION AND VISUALIZATION
# ============================================================================

print("Loading best model for evaluation...")
model = tf.keras.models.load_model('best_enhanced_unet_model.h5', custom_objects={
    'weighted_bce_dice_loss': weighted_bce_dice_loss,
    'dice_coef': dice_coef,
    'precision_metric': precision_metric,
    'recall_metric': recall_metric,
    'f1_metric': f1_metric,
    'SpectralAttention': SpectralAttention
})

print("Validation Metrics:")
val_results = model.evaluate(X_val, y_val, verbose=0)
print(f"Loss: {val_results[0]:.4f}")
print(f"Accuracy: {val_results[1]:.4f}")
print(f"Dice Coefficient: {val_results[2]:.4f}")
print(f"Precision: {val_results[3]:.4f}")
print(f"Recall: {val_results[4]:.4f}")
print(f"F1 Score: {val_results[5]:.4f}")

print("\nTest Metrics:")
test_results = model.evaluate(X_test, y_test, verbose=0)
print(f"Loss: {test_results[0]:.4f}")
print(f"Accuracy: {test_results[1]:.4f}")
print(f"Dice Coefficient: {test_results[2]:.4f}")
print(f"Precision: {test_results[3]:.4f}")
print(f"Recall: {test_results[4]:.4f}")
print(f"F1 Score: {test_results[5]:.4f}")

# Generate predictions
print("Generating predictions...")
val_predictions = model.predict(X_val, verbose=0)
test_predictions = model.predict(X_test, verbose=0)

# Apply threshold
val_predictions_binary = (val_predictions > 0.5).astype(np.float32)
test_predictions_binary = (test_predictions > 0.5).astype(np.float32)

# Visualization functions
def visualize_results(images, masks, predictions, num_samples=5):
    fig, axes = plt.subplots(num_samples, 4, figsize=(20, 5*num_samples))
    
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        # Original image
        axes[i, 0].imshow(images[i])
        axes[i, 0].set_title('Original Image')
        axes[i, 0].axis('off')
        
        # Ground truth mask
        axes[i, 1].imshow(masks[i].squeeze(), cmap='gray')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        # Prediction
        axes[i, 2].imshow(predictions[i].squeeze(), cmap='gray')
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')
        
        # Overlay
        axes[i, 3].imshow(images[i])
        axes[i, 3].imshow(predictions[i].squeeze(), cmap='jet', alpha=0.5)
        axes[i, 3].set_title('Overlay')
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig('enhanced_unet_segmentation_results.png', dpi=300, bbox_inches='tight')
    plt.show()

print("Visualizing validation results...")
visualize_results(X_val, y_val, val_predictions_binary, num_samples=min(5, len(X_val)))

print("Visualizing test results...")
visualize_results(X_test, y_test, test_predictions_binary, num_samples=min(5, len(X_test)))

# ============================================================================
# 11. PLOT TRAINING HISTORY
# ============================================================================

def plot_training_history(history):
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    metrics = ['loss', 'accuracy', 'dice_coef', 'precision_metric', 'recall_metric', 'f1_metric']
    titles = ['Loss', 'Accuracy', 'Dice Coefficient', 'Precision', 'Recall', 'F1 Score']
    
    for i, (metric, title) in enumerate(zip(metrics, titles)):
        row, col = i // 3, i % 3
        axes[row, col].plot(history.history[metric], label=f'Training {title}', linewidth=2)
        axes[row, col].plot(history.history[f'val_{metric}'], label=f'Validation {title}', linewidth=2)
        axes[row, col].set_title(f'{title} - Enhanced Spectral U-Net', fontsize=12, fontweight='bold')
        axes[row, col].set_xlabel('Epoch')
        axes[row, col].set_ylabel(title)
        axes[row, col].legend()
        axes[row, col].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('enhanced_unet_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history(history)

print("Enhanced Spectral U-Net with all novel components completed successfully!")

***3. Spectral attention with post preprocessing***

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score, jaccard_score
from sklearn.utils.class_weight import compute_class_weight

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configuration
IMG_HEIGHT = 256
IMG_WIDTH = 256
BATCH_SIZE = 8
EPOCHS = 50
PATIENCE = 15

# Paths to your data
TRAIN_IMAGE_DIR = '/kaggle/input/microplastic-s/training/training/Training_original'
TRAIN_MASK_DIR = '/kaggle/input/microplastic-s/training/training/Finalmasks'
VAL_IMAGE_DIR = '/kaggle/input/microplastic-s/val/val/Original'
VAL_MASK_DIR = '/kaggle/input/microplastic-s/val/val/val_100masks'
TEST_IMAGE_DIR = '/kaggle/input/microplastic-s/Testing/Testing/Original'
TEST_MASK_DIR = '/kaggle/input/microplastic-s/Testing/Testing/testing_100masks'

# ============================================================================
# 1. COMPREHENSIVE PREPROCESSING FUNCTIONS
# ============================================================================

def advanced_preprocess_image(image):
    """Enhanced preprocessing pipeline with multiple techniques"""
    if len(image.shape) == 3:
        image_uint8 = (image * 255).astype(np.uint8)
        
        # 1. CLAHE for contrast enhancement
        lab = cv2.cvtColor(image_uint8, cv2.COLOR_RGB2LAB)
        lab_planes = list(cv2.split(lab))
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        lab_planes[0] = clahe.apply(lab_planes[0])
        lab = cv2.merge(lab_planes)
        image_uint8 = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        
        # 2. Gaussian blur for noise reduction
        image_uint8 = cv2.GaussianBlur(image_uint8, (3, 3), 0)
        
        # 3. Median filtering for salt-and-pepper noise
        image_uint8 = cv2.medianBlur(image_uint8, 3)
        
        # Convert back to float
        image_processed = image_uint8.astype(np.float32) / 255.0
        
        # 4. Color normalization
        normalized = np.zeros_like(image_processed)
        for channel in range(3):
            channel_data = image_processed[:, :, channel]
            normalized[:, :, channel] = (channel_data - np.mean(channel_data)) / (np.std(channel_data) + 1e-8)
        
        # Adaptive normalization
        if np.std(normalized) > 0.1:
            normalized = (normalized - np.min(normalized)) / (np.max(normalized) - np.min(normalized))
        
        return np.clip(normalized, 0, 1)
    else:
        return image

def enhance_microplastic_contrast(image, mask):
    """Enhanced contrast specifically around microplastic regions"""
    if len(mask.shape) == 3:
        mask_binary = mask.squeeze() > 0.5
    else:
        mask_binary = mask > 0.5
    
    image_uint8 = (image * 255).astype(np.uint8)
    
    # Apply sharpening to entire image
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(image_uint8, -1, kernel)
    
    sharpened = sharpened.astype(np.float32) / 255.0
    image_float = image_uint8.astype(np.float32) / 255.0
    
    # Blend based on mask
    enhanced_image = np.where(mask_binary[..., np.newaxis], sharpened, image_float)
    
    return enhanced_image

# ============================================================================
# 2. SPECTRAL ATTENTION BLOCKS
# ============================================================================

class SpectralAttention(layers.Layer):
    """Spectral attention for frequency-aware feature processing"""
    def __init__(self, filters, reduction_ratio=16, **kwargs):
        super(SpectralAttention, self).__init__(**kwargs)
        self.filters = filters
        self.reduction_ratio = reduction_ratio
        
        self.global_avg_pool = layers.GlobalAveragePooling2D()
        self.global_max_pool = layers.GlobalMaxPooling2D()
        
    def build(self, input_shape):
        self.fc1 = layers.Dense(self.filters // self.reduction_ratio, activation='relu')
        self.fc2 = layers.Dense(self.filters, activation='sigmoid')
        super(SpectralAttention, self).build(input_shape)
        
    def call(self, inputs):
        # Channel attention via both avg and max pooling
        avg_pool = self.global_avg_pool(inputs)
        max_pool = self.global_max_pool(inputs)
        
        # Shared MLP
        avg_out = self.fc2(self.fc1(avg_pool))
        max_out = self.fc2(self.fc1(max_pool))
        
        # Combine attention maps
        channel_attention = tf.sigmoid(avg_out + max_out)
        channel_attention = tf.reshape(channel_attention, [-1, 1, 1, self.filters])
        
        return inputs * channel_attention
    
    def get_config(self):
        config = super(SpectralAttention, self).get_config()
        config.update({
            'filters': self.filters,
            'reduction_ratio': self.reduction_ratio
        })
        return config

# ============================================================================
# 3. ENHANCED RESIDUAL BLOCKS WITH SPECTRAL ATTENTION
# ============================================================================

def residual_block(input_tensor, num_filters):
    """Simple residual block without downsampling"""
    # Main path
    x = layers.Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    x = layers.Conv2D(num_filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Shortcut path
    shortcut = input_tensor
    if input_tensor.shape[-1] != num_filters:
        shortcut = layers.Conv2D(num_filters, (1, 1), padding='same')(input_tensor)
        shortcut = layers.BatchNormalization()(shortcut)
    
    # Add shortcut to main path
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    return x

def enhanced_residual_block(input_tensor, num_filters, use_spectral_attention=True):
    """Enhanced residual block with spectral attention"""
    # Main path
    x = layers.Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    x = layers.Conv2D(num_filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Apply spectral attention
    if use_spectral_attention:
        x = SpectralAttention(num_filters)(x)
    
    # Shortcut path
    shortcut = input_tensor
    if input_tensor.shape[-1] != num_filters:
        shortcut = layers.Conv2D(num_filters, (1, 1), padding='same')(input_tensor)
        shortcut = layers.BatchNormalization()(shortcut)
    
    # Add residual connection
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    return x

# ============================================================================
# 4. ENHANCED U-NET ARCHITECTURE WITH NOVEL COMPONENTS
# ============================================================================

def build_enhanced_unet(input_shape=(256, 256, 3)):
    """Enhanced U-Net with spectral attention and residual connections"""
    inputs = layers.Input(shape=input_shape)
    
    # Encoder
    # Level 1
    x1 = layers.Conv2D(64, (3, 3), padding='same')(inputs)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Activation('relu')(x1)
    x1 = SpectralAttention(64)(x1)  # Spectral attention
    p1 = layers.MaxPooling2D((2, 2))(x1)
    
    # Level 2
    x2 = enhanced_residual_block(p1, 128, use_spectral_attention=True)
    p2 = layers.MaxPooling2D((2, 2))(x2)
    
    # Level 3
    x3 = enhanced_residual_block(p2, 256, use_spectral_attention=True)
    p3 = layers.MaxPooling2D((2, 2))(x3)
    
    # Level 4
    x4 = enhanced_residual_block(p3, 512, use_spectral_attention=True)
    p4 = layers.MaxPooling2D((2, 2))(x4)
    
    # Bridge
    bridge = enhanced_residual_block(p4, 1024, use_spectral_attention=True)
    
    # Decoder
    # Level 4 up
    u4 = layers.Conv2DTranspose(512, (2, 2), strides=2, padding='same')(bridge)
    u4 = layers.concatenate([u4, x4])
    u4 = enhanced_residual_block(u4, 512, use_spectral_attention=True)
    
    # Level 3 up
    u3 = layers.Conv2DTranspose(256, (2, 2), strides=2, padding='same')(u4)
    u3 = layers.concatenate([u3, x3])
    u3 = enhanced_residual_block(u3, 256, use_spectral_attention=True)
    
    # Level 2 up
    u2 = layers.Conv2DTranspose(128, (2, 2), strides=2, padding='same')(u3)
    u2 = layers.concatenate([u2, x2])
    u2 = enhanced_residual_block(u2, 128, use_spectral_attention=True)
    
    # Level 1 up
    u1 = layers.Conv2DTranspose(64, (2, 2), strides=2, padding='same')(u2)
    u1 = layers.concatenate([u1, x1])
    u1 = enhanced_residual_block(u1, 64, use_spectral_attention=True)
    
    # Output
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(u1)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='Enhanced_Spectral_UNet')
    return model

# ============================================================================
# 5. MORPHOLOGICAL POST-PROCESSING FUNCTIONS
# ============================================================================

def apply_morphological_operations(predictions, kernel_size=3):
    """
    Apply morphological operations to clean up predictions
    """
    processed_predictions = []
    
    for pred in predictions:
        # Convert to binary
        binary_pred = (pred.squeeze() > 0.5).astype(np.uint8)
        
        # 1. Remove small noise (opening)
        kernel = np.ones((kernel_size, kernel_size), np.uint8)
        cleaned = cv2.morphologyEx(binary_pred, cv2.MORPH_OPEN, kernel)
        
        # 2. Fill small holes (closing)
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)
        
        # 3. Remove very small connected components
        cleaned = remove_small_objects(cleaned, min_size=10)
        
        processed_predictions.append(cleaned)
    
    return np.array(processed_predictions)[..., np.newaxis].astype(np.float32)

def remove_small_objects(binary_mask, min_size=50):
    """
    Remove connected components smaller than the specified size
    """
    # Find connected components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    
    # Create output image
    output = np.zeros_like(binary_mask)
    
    # Iterate through components (skip background - label 0)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_size:
            output[labels == i] = 1
    
    return output

def fill_holes(binary_mask):
    """
    Fill holes in binary masks
    """
    # Copy the thresholded image
    im_floodfill = binary_mask.copy()
    
    # Mask used for flooding
    h, w = binary_mask.shape[:2]
    mask = np.zeros((h+2, w+2), np.uint8)
    
    # Floodfill from point (0, 0)
    cv2.floodFill(im_floodfill, mask, (0,0), 1)
    
    # Invert floodfilled image
    im_floodfill_inv = cv2.bitwise_not(im_floodfill)
    
    # Combine the two images to get the foreground
    im_out = binary_mask | im_floodfill_inv
    
    return im_out

def microplastic_specific_postprocessing(predictions, original_images=None):
    """
    Microplastic-specific postprocessing based on physical properties
    """
    processed = []
    
    for i, pred in enumerate(predictions):
        binary_pred = (pred.squeeze() > 0.5).astype(np.uint8)
        
        # 1. Size-based filtering (microplastics have size constraints)
        size_filtered = filter_by_size(binary_pred, min_area=5, max_area=5000)
        
        # 2. Shape-based filtering (compactness, circularity)
        shape_filtered = filter_by_shape(size_filtered, min_circularity=0.1, max_elongation=5.0)
        
        # 3. Apply morphological smoothing
        kernel = np.ones((2, 2), np.uint8)
        smoothed = cv2.morphologyEx(shape_filtered, cv2.MORPH_CLOSE, kernel)
        
        # 4. Remove artifacts near image borders
        border_cleaned = remove_border_objects(smoothed, border_width=5)
        
        processed.append(border_cleaned)
    
    return np.array(processed)[..., np.newaxis].astype(np.float32)

def filter_by_size(binary_mask, min_area=5, max_area=5000):
    """
    Filter objects based on size constraints typical for microplastics
    """
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    
    output = np.zeros_like(binary_mask)
    
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if min_area <= area <= max_area:
            output[labels == i] = 1
    
    return output

def filter_by_shape(binary_mask, min_circularity=0.1, max_elongation=5.0):
    """
    Filter objects based on shape characteristics
    """
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    
    output = np.zeros_like(binary_mask)
    
    for i in range(1, num_labels):
        # Get object mask
        obj_mask = (labels == i).astype(np.uint8)
        
        # Calculate shape properties
        area = stats[i, cv2.CC_STAT_AREA]
        
        # Find contours for perimeter calculation
        contours, _ = cv2.findContours(obj_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            perimeter = cv2.arcLength(contours[0], True)
        else:
            perimeter = 0
        
        # Circularity
        if perimeter > 0:
            circularity = 4 * np.pi * area / (perimeter * perimeter)
        else:
            circularity = 0
        
        # Elongation (aspect ratio of bounding box)
        x, y, w, h, _ = stats[i]
        if h > 0:
            elongation = w / h
            if elongation < 1:
                elongation = 1 / elongation
        else:
            elongation = 1.0
        
        # Keep objects that meet shape criteria
        if circularity >= min_circularity and elongation <= max_elongation:
            output[labels == i] = 1
    
    return output

def remove_border_objects(binary_mask, border_width=5):
    """
    Remove objects touching the image borders
    """
    h, w = binary_mask.shape
    
    # Create border mask
    border_mask = np.zeros_like(binary_mask)
    border_mask[:border_width, :] = 1  # Top
    border_mask[-border_width:, :] = 1  # Bottom
    border_mask[:, :border_width] = 1  # Left
    border_mask[:, -border_width:] = 1  # Right
    
    # Find objects touching borders
    border_objects = binary_mask & border_mask
    
    # Remove border-touching objects
    num_labels, labels = cv2.connectedComponents(border_objects, connectivity=8)
    
    output = binary_mask.copy()
    for i in range(1, num_labels):
        output[labels == i] = 0
    
    return output

# ============================================================================
# 6. DATA LOADING WITH PREPROCESSING
# ============================================================================

def load_data_from_dirs(image_dir, mask_dir, preprocess=True, enhance_contrast=True):
    image_paths = sorted([os.path.join(image_dir, fname) for fname in os.listdir(image_dir) 
                         if fname.endswith(('.png', '.jpg', '.jpeg'))])
    mask_paths = sorted([os.path.join(mask_dir, fname) for fname in os.listdir(mask_dir) 
                        if fname.endswith(('.png', '.jpg', '.jpeg'))])
    
    images = []
    masks = []
    
    for img_path, mask_path in zip(image_paths, mask_paths):
        # Load image
        img = Image.open(img_path).convert('RGB')
        img = img.resize((IMG_WIDTH, IMG_HEIGHT))
        img_array = np.array(img, dtype=np.float32) / 255.0
        
        # Load mask - ensure proper binarization
        mask = Image.open(mask_path).convert('L')
        mask = mask.resize((IMG_WIDTH, IMG_HEIGHT))
        mask_array = np.array(mask, dtype=np.float32)
        mask_array = (mask_array > 0.5).astype(np.float32)
        mask_array = np.expand_dims(mask_array, axis=-1)
        
        # Apply preprocessing
        if preprocess:
            img_array = advanced_preprocess_image(img_array)
        
        # Enhance contrast around microplastics
        if enhance_contrast:
            img_array = enhance_microplastic_contrast(img_array, mask_array)
        
        images.append(img_array)
        masks.append(mask_array)
    
    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

# Load data with preprocessing
print("Loading and preprocessing data...")
X_train, y_train = load_data_from_dirs(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR, preprocess=True, enhance_contrast=True)
X_val, y_val = load_data_from_dirs(VAL_IMAGE_DIR, VAL_MASK_DIR, preprocess=True, enhance_contrast=True)
X_test, y_test = load_data_from_dirs(TEST_IMAGE_DIR, TEST_MASK_DIR, preprocess=True, enhance_contrast=True)

print(f"Training data: {X_train.shape}, {y_train.shape}")
print(f"Positive pixels in training: {np.sum(y_train > 0) / y_train.size:.6f}")

# ============================================================================
# 7. CLASS WEIGHTS AND LOSS FUNCTIONS
# ============================================================================

def calculate_class_weights(masks):
    masks_flat = masks.flatten()
    class_weights = compute_class_weight('balanced', classes=[0, 1], y=masks_flat)
    return {0: class_weights[0], 1: class_weights[1]}

class_weights = calculate_class_weights(y_train)
print(f"Class weights: {class_weights}")

def weighted_bce_dice_loss(y_true, y_pred):
    # Calculate class weights for this batch
    y_true_flat = K.flatten(y_true)
    y_pred_flat = K.flatten(y_pred)
    
    # Binary cross entropy with class weighting
    bce = tf.keras.losses.binary_crossentropy(y_true_flat, y_pred_flat)
    
    # Apply class weights to BCE
    class_weight_0 = tf.constant(class_weights[0], dtype=tf.float32)
    class_weight_1 = tf.constant(class_weights[1], dtype=tf.float32)
    
    # Create weight tensor based on true labels
    weights = tf.where(tf.equal(y_true_flat, 1.0), class_weight_1, class_weight_0)
    weighted_bce = tf.reduce_mean(weights * bce)
    
    # Dice loss
    intersection = tf.reduce_sum(y_true_flat * y_pred_flat)
    union = tf.reduce_sum(y_true_flat) + tf.reduce_sum(y_pred_flat)
    dice_loss = 1.0 - (2.0 * intersection + 1.0) / (union + 1.0)
    
    return weighted_bce + dice_loss

def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

# Enhanced metrics with thresholding
def precision_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    # Threshold predictions
    y_pred_thresholded = tf.cast(y_pred > 0.5, tf.float32)
    
    true_positives = tf.reduce_sum(y_true * y_pred_thresholded)
    predicted_positives = tf.reduce_sum(y_pred_thresholded)
    
    precision = true_positives / (predicted_positives + tf.keras.backend.epsilon())
    return precision

def recall_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    # Threshold predictions
    y_pred_thresholded = tf.cast(y_pred > 0.5, tf.float32)
    
    true_positives = tf.reduce_sum(y_true * y_pred_thresholded)
    actual_positives = tf.reduce_sum(y_true)
    
    recall = true_positives / (actual_positives + tf.keras.backend.epsilon())
    return recall

def f1_metric(y_true, y_pred):
    prec = precision_metric(y_true, y_pred)
    rec = recall_metric(y_true, y_pred)
    return 2 * ((prec * rec) / (prec + rec + tf.keras.backend.epsilon()))

# ============================================================================
# 8. BUILD AND COMPILE THE ENHANCED MODEL
# ============================================================================

print("Building Enhanced Spectral U-Net...")
model = build_enhanced_unet()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=weighted_bce_dice_loss,
    metrics=['accuracy', dice_coef, precision_metric, recall_metric, f1_metric]
)

model.summary()

# ============================================================================
# 9. DATA GENERATOR
# ============================================================================

class MicroplasticDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, y, batch_size=8, shuffle=True, augment=False):
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.indexes = np.arange(len(X))
        self.on_epoch_end()
    
    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))
    
    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        
        X_batch = self.X[indexes]
        y_batch = self.y[indexes]
        
        if self.augment:
            X_batch, y_batch = self.augment_batch(X_batch, y_batch)
        
        return X_batch, y_batch
    
    def augment_batch(self, X_batch, y_batch):
        augmented_X = []
        augmented_y = []
        
        for i in range(len(X_batch)):
            img = X_batch[i].copy()
            mask = y_batch[i].copy()
            
            # Random horizontal flip (50% chance)
            if np.random.random() > 0.5:
                img = np.fliplr(img)
                mask = np.fliplr(mask)
            
            # Random vertical flip (50% chance)
            if np.random.random() > 0.5:
                img = np.flipud(img)
                mask = np.flipud(mask)
            
            # Random rotation (50% chance)
            if np.random.random() > 0.5:
                k = np.random.randint(1, 4)  # 90, 180, or 270 degrees
                img = np.rot90(img, k)
                mask = np.rot90(mask, k)
            
            # Random brightness adjustment (50% chance)
            if np.random.random() > 0.5:
                factor = np.random.uniform(0.7, 1.3)
                img = np.clip(img * factor, 0, 1)
            
            # Random contrast adjustment (50% chance)
            if np.random.random() > 0.5:
                factor = np.random.uniform(0.7, 1.3)
                mean = np.mean(img, axis=(0, 1), keepdims=True)
                img = np.clip((img - mean) * factor + mean, 0, 1)
            
            # Random Gaussian noise (30% chance)
            if np.random.random() > 0.7:
                noise = np.random.normal(0, 0.02, img.shape)
                img = np.clip(img + noise, 0, 1)
            
            augmented_X.append(img)
            augmented_y.append(mask)
        
        return np.array(augmented_X), np.array(augmented_y)
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# ============================================================================
# 10. TRAINING WITH EARLY STOPPING
# ============================================================================

train_generator = MicroplasticDataGenerator(X_train, y_train, batch_size=BATCH_SIZE, 
                                          shuffle=True, augment=True)
val_generator = MicroplasticDataGenerator(X_val, y_val, batch_size=BATCH_SIZE, 
                                        shuffle=False, augment=False)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_dice_coef', 
        patience=PATIENCE, 
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_dice_coef', 
        factor=0.5, 
        patience=8, 
        min_lr=1e-7,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_enhanced_unet_model.h5', 
        monitor='val_dice_coef', 
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

print("Starting training...")
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks,
    verbose=1
)

# ============================================================================
# 11. POST-PROCESSING EVALUATION AND VISUALIZATION
# ============================================================================

print("Loading best model for evaluation...")
model = tf.keras.models.load_model('best_enhanced_unet_model.h5', custom_objects={
    'weighted_bce_dice_loss': weighted_bce_dice_loss,
    'dice_coef': dice_coef,
    'precision_metric': precision_metric,
    'recall_metric': recall_metric,
    'f1_metric': f1_metric,
    'SpectralAttention': SpectralAttention
})

print("Validation Metrics:")
val_results = model.evaluate(X_val, y_val, verbose=0)
print(f"Loss: {val_results[0]:.4f}")
print(f"Accuracy: {val_results[1]:.4f}")
print(f"Dice Coefficient: {val_results[2]:.4f}")
print(f"Precision: {val_results[3]:.4f}")
print(f"Recall: {val_results[4]:.4f}")
print(f"F1 Score: {val_results[5]:.4f}")

print("\nTest Metrics:")
test_results = model.evaluate(X_test, y_test, verbose=0)
print(f"Loss: {test_results[0]:.4f}")
print(f"Accuracy: {test_results[1]:.4f}")
print(f"Dice Coefficient: {test_results[2]:.4f}")
print(f"Precision: {test_results[3]:.4f}")
print(f"Recall: {test_results[4]:.4f}")
print(f"F1 Score: {test_results[5]:.4f}")

# Generate predictions
print("Generating predictions...")
val_predictions = model.predict(X_val, verbose=0)
test_predictions = model.predict(X_test, verbose=0)

# Apply basic threshold
val_predictions_binary = (val_predictions > 0.5).astype(np.float32)
test_predictions_binary = (test_predictions > 0.5).astype(np.float32)

# APPLY MORPHOLOGICAL POST-PROCESSING
print("Applying morphological post-processing...")
val_predictions_processed = microplastic_specific_postprocessing(val_predictions)
test_predictions_processed = microplastic_specific_postprocessing(test_predictions)

# Also apply basic morphological operations
val_predictions_morph = apply_morphological_operations(val_predictions)
test_predictions_morph = apply_morphological_operations(test_predictions)

# ============================================================================
# 12. POST-PROCESSING COMPARISON AND EVALUATION
# ============================================================================

def numpy_dice_coef(y_true, y_pred, smooth=1.0):
    """Dice coefficient for numpy arrays"""
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

def visualize_postprocessing_comparison(images, masks, raw_pred, morph_pred, advanced_pred, num_samples=3):
    """
    Compare raw predictions vs post-processed predictions
    """
    fig, axes = plt.subplots(num_samples, 6, figsize=(24, 4*num_samples))
    
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        col = 0
        
        # Original image
        axes[i, col].imshow(images[i])
        axes[i, col].set_title('Original Image')
        axes[i, col].axis('off')
        col += 1
        
        # Ground truth
        axes[i, col].imshow(masks[i].squeeze(), cmap='gray')
        axes[i, col].set_title('Ground Truth')
        axes[i, col].axis('off')
        col += 1
        
        # Raw prediction
        axes[i, col].imshow(raw_pred[i].squeeze(), cmap='gray')
        axes[i, col].set_title('Raw Prediction')
        axes[i, col].axis('off')
        col += 1
        
        # Basic morphological
        axes[i, col].imshow(morph_pred[i].squeeze(), cmap='gray')
        axes[i, col].set_title('Basic Morphological')
        axes[i, col].axis('off')
        col += 1
        
        # Advanced post-processing
        axes[i, col].imshow(advanced_pred[i].squeeze(), cmap='gray')
        axes[i, col].set_title('Advanced Processing')
        axes[i, col].axis('off')
        col += 1
        
        # Overlay comparison
        axes[i, col].imshow(images[i])
        axes[i, col].imshow(advanced_pred[i].squeeze(), cmap='jet', alpha=0.5)
        axes[i, col].set_title('Final Overlay')
        axes[i, col].axis('off')
    
    plt.tight_layout()
    plt.savefig('postprocessing_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

def evaluate_postprocessing_effectiveness(true_masks, raw_pred, processed_pred):
    """
    Compare metrics before and after post-processing
    """
    def calculate_metrics(y_true, y_pred):
        y_true_flat = y_true.flatten().astype(np.int32)
        y_pred_flat = y_pred.flatten().astype(np.int32)
        
        # Use numpy version for evaluation
        dice = numpy_dice_coef(y_true_flat, y_pred_flat)
        precision = precision_score(y_true_flat, y_pred_flat, zero_division=0)
        recall = recall_score(y_true_flat, y_pred_flat, zero_division=0)
        f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)
        
        return dice, precision, recall, f1
    
    # Calculate metrics for raw predictions
    raw_dice, raw_precision, raw_recall, raw_f1 = calculate_metrics(true_masks, raw_pred)
    
    # Calculate metrics for processed predictions
    proc_dice, proc_precision, proc_recall, proc_f1 = calculate_metrics(true_masks, processed_pred)
    
    print("=== POST-PROCESSING EFFECTIVENESS ===")
    print(f"{'Metric':<12} {'Raw':<8} {'Processed':<10} {'Improvement':<12}")
    print(f"{'Dice':<12} {raw_dice:.4f}    {proc_dice:.4f}      {proc_dice-raw_dice:+.4f}")
    print(f"{'Precision':<12} {raw_precision:.4f}    {proc_precision:.4f}      {proc_precision-raw_precision:+.4f}")
    print(f"{'Recall':<12} {raw_recall:.4f}    {proc_recall:.4f}      {proc_recall-raw_recall:+.4f}")
    print(f"{'F1-Score':<12} {raw_f1:.4f}    {proc_f1:.4f}      {proc_f1-raw_f1:+.4f}")
    
    return {
        'raw': {'dice': raw_dice, 'precision': raw_precision, 'recall': raw_recall, 'f1': raw_f1},
        'processed': {'dice': proc_dice, 'precision': proc_precision, 'recall': proc_recall, 'f1': proc_f1}
    }

print("Visualizing post-processing comparison...")
visualize_postprocessing_comparison(
    X_val[:3], y_val[:3], 
    val_predictions_binary[:3], 
    val_predictions_morph[:3],
    val_predictions_processed[:3],
    num_samples=min(3, len(X_val))
)

print("Evaluating post-processing effectiveness...")
val_metrics = evaluate_postprocessing_effectiveness(y_val, val_predictions_binary, val_predictions_processed)
test_metrics = evaluate_postprocessing_effectiveness(y_test, test_predictions_binary, test_predictions_processed)

# ============================================================================
# 13. FINAL VISUALIZATION
# ============================================================================

def visualize_results(images, masks, predictions, num_samples=5):
    fig, axes = plt.subplots(num_samples, 4, figsize=(20, 5*num_samples))
    
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        # Original image
        axes[i, 0].imshow(images[i])
        axes[i, 0].set_title('Original Image')
        axes[i, 0].axis('off')
        
        # Ground truth mask
        axes[i, 1].imshow(masks[i].squeeze(), cmap='gray')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        # Prediction
        axes[i, 2].imshow(predictions[i].squeeze(), cmap='gray')
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')
        
        # Overlay
        axes[i, 3].imshow(images[i])
        axes[i, 3].imshow(predictions[i].squeeze(), cmap='jet', alpha=0.5)
        axes[i, 3].set_title('Overlay')
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig('enhanced_unet_segmentation_results.png', dpi=300, bbox_inches='tight')
    plt.show()

print("Visualizing final processed results...")
visualize_results(X_val, y_val, val_predictions_processed, num_samples=min(5, len(X_val)))
visualize_results(X_test, y_test, test_predictions_processed, num_samples=min(5, len(X_test)))

# ============================================================================
# 14. PLOT TRAINING HISTORY
# ============================================================================

def plot_training_history(history):
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    metrics = ['loss', 'accuracy', 'dice_coef', 'precision_metric', 'recall_metric', 'f1_metric']
    titles = ['Loss', 'Accuracy', 'Dice Coefficient', 'Precision', 'Recall', 'F1 Score']
    
    for i, (metric, title) in enumerate(zip(metrics, titles)):
        row, col = i // 3, i % 3
        axes[row, col].plot(history.history[metric], label=f'Training {title}', linewidth=2)
        axes[row, col].plot(history.history[f'val_{metric}'], label=f'Validation {title}', linewidth=2)
        axes[row, col].set_title(f'{title} - Enhanced Spectral U-Net', fontsize=12, fontweight='bold')
        axes[row, col].set_xlabel('Epoch')
        axes[row, col].set_ylabel(title)
        axes[row, col].legend()
        axes[row, col].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('enhanced_unet_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history(history)

print("Enhanced Spectral U-Net with morphological post-processing completed successfully!")

***FCN-VGG16***